# Assignment: Build a small OSM and census workflow

## Goal

Build an end-to-end workflow that retrieves or loads an urban feature dataset, saves it, cleans it, connects it to Toronto census areas, and produces a simple mapped measure.

The core version uses **libraries**. If you are comfortable, you may adapt the OSM query to another supported feature type such as schools or parks.

There is no formal submission step. The goal is to produce a notebook that runs from top to bottom and documents what you did.

In [ ]:
from pathlib import Path

import geopandas as gpd
import pandas as pd
import matplotlib.pyplot as plt
import osmnx as ox

## Part 1: Define the workflow settings

Keep your place name, tags, and paths together near the top.

In [ ]:
PLACE = "Toronto, Ontario, Canada"
TAGS = {"amenity": "library"}

RAW_DIR = Path("../data/raw")
PROCESSED_DIR = Path("../data/processed")
OUTPUT_DIR = Path("../output")

CENSUS_FILE = RAW_DIR / "toronto_census_areas.geojson"
OSM_BACKUP_FILE = Path(
    "../data/backup/toronto_osm_libraries_live_backup.gpkg"
)

PRACTICE_FILE = Path(
    "../data/practice/toronto_library_points_practice.geojson"
)

RAW_OSM_FILE = RAW_DIR / "assignment_osm_features.gpkg"
CLEAN_FILE = PROCESSED_DIR / "assignment_features_clean.gpkg"
COMBINED_FILE = PROCESSED_DIR / "assignment_census_features.gpkg"
SUMMARY_FILE = OUTPUT_DIR / "assignment_summary.csv"

## Part 2: Create the required folders

Use Python to ensure the raw, processed, and output directories exist.

**Hint:** `mkdir(parents=True, exist_ok=True)`

In [ ]:
RAW_DIR.mkdir(parents=True, exist_ok=True)
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

## Part 3: Retrieve the feature data

Try the live OSMnx query first:

```python
features = ox.features_from_place(
    PLACE,
    tags=TAGS
)
```

If the service is unavailable, switch to the saved backup:

```text
OSM_BACKUP_FILE
```

This is a saved result from a successful live OSMnx library query.

`PRACTICE_FILE` is available as a second fallback, but it comes from the course Toronto Public Library dataset and is not an OSM extract.

Record which source you used in your final workflow description.

In [ ]:
USE_LIVE_QUERY = False

if USE_LIVE_QUERY:
    features = ox.features_from_place(
        PLACE,
        tags=TAGS
    )
    data_source = "live OpenStreetMap query"
else:
    features = gpd.read_file(
        OSM_BACKUP_FILE
    )
    data_source = "saved live OSM classroom backup"

print("Data source:", data_source)

## Part 4: Inspect the raw data

Before cleaning, inspect:

- shape;
- CRS;
- geometry types;
- columns;
- first records;
- missing names if available.

Write one or two sentences describing what you notice.

In [ ]:
print("Shape:", features.shape)
print("CRS:", features.crs)

print("\nGeometry types:")
print(features.geometry.geom_type.value_counts())

print("\nColumns:")
print(features.columns.tolist())

display(features.head())

if "name" in features.columns:
    print("\nMissing names:", features["name"].isna().sum())

The feature data contains mapped library locations with a mixture of OSM attributes and geometry types. Many OSM tag columns are optional, so some fields contain substantial missing data; the analysis therefore keeps only the identifiers, name, and geometry needed for the spatial workflow.

## Part 5: Save the raw feature data

Save the retrieved GeoDataFrame to `RAW_OSM_FILE`.

This is an important workflow checkpoint: later code should be able to work from the saved file without repeating the query.

In [ ]:
features.to_file(
    RAW_OSM_FILE,
    driver="GPKG"
)

print("Saved:", RAW_OSM_FILE)

## Part 6: Reopen and clean the saved data

Reopen `RAW_OSM_FILE`.

Then:

- reset the index when useful;
- retain a small set of identifying columns and geometry;
- store the cleaned result as `features_clean`;
- save it to `CLEAN_FILE`.

Useful columns may include:

```text
element_type
osmid
name
geometry
```

Not every source will contain all of them, so inspect before selecting.

In [ ]:
saved_features = gpd.read_file(
    RAW_OSM_FILE
)

features_clean = saved_features.reset_index()

desired_columns = [
    "element_type",
    "osmid",
    "name",
    "geometry"
]

columns_to_keep = []

for column in desired_columns:
    if column in features_clean.columns:
        columns_to_keep.append(column)

features_clean = features_clean[
    columns_to_keep
].copy()

features_clean.to_file(
    CLEAN_FILE,
    driver="GPKG"
)

display(features_clean.head())

## Part 7: Load the census geography

Load `CENSUS_FILE` and inspect:

- shape;
- CRS;
- geometry type;
- `ADAUID`;
- `pop_2021_count`;
- `hh_no_veh_pct`.

One row represents one Toronto census geography.

In [ ]:
census = gpd.read_file(
    CENSUS_FILE
)

print("Shape:", census.shape)
print("CRS:", census.crs)
print("\nGeometry types:")
print(census.geometry.geom_type.value_counts())

display(
    census[
        [
            "ADAUID",
            "pop_2021_count",
            "hh_no_veh_pct",
            "geometry"
        ]
    ].head()
)

## Part 8: Prepare the spatial join

Put the feature data in the same CRS as the census polygons.

If the feature geometries include polygons, use `representative_point()` so every feature has one point location.

Store the result as `feature_points`.

In [ ]:
feature_points = features_clean.to_crs(
    census.crs
).copy()

feature_points["geometry"] = (
    feature_points.geometry.representative_point()
)

## Part 9: Spatially join features to census areas

Attach:

```text
ADAUID
pop_2021_count
hh_no_veh_pct
```

using `gpd.sjoin()` with:

```python
how="left"
predicate="within"
```

Then check the result before continuing.

In [ ]:
joined = gpd.sjoin(
    feature_points,
    census[
        [
            "ADAUID",
            "pop_2021_count",
            "hh_no_veh_pct",
            "geometry"
        ]
    ],
    how="left",
    predicate="within"
)

print("Rows before join:", len(feature_points))
print("Rows after join:", len(joined))
print(
    "Missing ADAUID:",
    joined["ADAUID"].isna().sum()
)

## Part 10: Count features by census area

Create one table containing:

```text
ADAUID
feature_count
```

Then left-merge those counts back to all census polygons.

After checking the merge, replace missing feature counts with zero.

In [ ]:
feature_counts = (
    joined
    .groupby("ADAUID")
    .size()
    .reset_index(name="feature_count")
)

combined = census.merge(
    feature_counts,
    on="ADAUID",
    how="left"
)

combined["feature_count"] = (
    combined["feature_count"].fillna(0)
)

print("Rows after merge:", len(combined))
print(
    "Missing feature counts:",
    combined["feature_count"].isna().sum()
)

## Part 11: Create one adjusted measure

Calculate:

```text
features_per_10000
```

using:

```text
feature_count / pop_2021_count × 10,000
```

Check whether any population values are missing or zero before interpreting the result.

In [ ]:
combined["features_per_10000"] = (
    combined["feature_count"]
    / combined["pop_2021_count"]
    * 10000
)

combined[
    [
        "ADAUID",
        "feature_count",
        "pop_2021_count",
        "features_per_10000"
    ]
].head()

## Part 12: Inspect the highest values

Display the five census areas with the highest:

1. raw feature count;
2. features per 10,000 residents.

Do the same areas appear in both lists?

In [ ]:
print("Highest raw feature counts:")
display(
    combined[
        ["ADAUID", "feature_count"]
    ]
    .sort_values(
        "feature_count",
        ascending=False
    )
    .head(5)
)

print("Highest features per 10,000 residents:")
display(
    combined[
        ["ADAUID", "features_per_10000"]
    ]
    .sort_values(
        "features_per_10000",
        ascending=False
    )
    .head(5)
)

## Part 13: Create a map

Create a choropleth of `features_per_10000`.

Include:

- a title;
- legend;
- census boundaries;
- hidden axes.

In [ ]:
ax = combined.plot(
    column="features_per_10000",
    legend=True,
    edgecolor="white",
    linewidth=0.2,
    figsize=(8, 8)
)

ax.set_title(
    "Libraries per 10,000 residents by Toronto census area"
)
ax.set_axis_off()

## Part 14: Save the final outputs

Save:

- the combined GeoDataFrame to `COMBINED_FILE`;
- a small CSV summary to `SUMMARY_FILE`.

The summary should contain at least:

```text
ADAUID
pop_2021_count
hh_no_veh_pct
feature_count
features_per_10000
```

In [ ]:
combined.to_file(
    COMBINED_FILE,
    driver="GPKG"
)

combined[
    [
        "ADAUID",
        "pop_2021_count",
        "hh_no_veh_pct",
        "feature_count",
        "features_per_10000"
    ]
].to_csv(
    SUMMARY_FILE,
    index=False
)

print("Saved:", COMBINED_FILE)
print("Saved:", SUMMARY_FILE)

## Part 15: Describe the workflow

Write approximately one paragraph explaining:

- the urban feature you examined;
- the OSM tag used;
- whether live OSM data or the practice fallback was used;
- why the raw data was saved;
- the main cleaning step;
- how features were connected to census areas;
- what the final map represents;
- one important limitation.

Possible limitations include incomplete OSM coverage, feature capacity, census-area size, travel distance, population need, and the fact that residents may use facilities outside their home census area.

This workflow examines libraries across Toronto using the OpenStreetMap tag `amenity=library`. It first attempts a live OSMnx request and uses the saved live OSM classroom backup if the service is unavailable. The retrieved data is saved before cleaning so the external request does not need to be repeated, then reopened and reduced to the fields needed for analysis. Library geometries are converted to representative points and spatially joined to Toronto census polygons, counted by census area, and converted to a rate per 10,000 residents. The final map therefore represents mapped library locations relative to residential population. An important limitation is that this does not measure library capacity, travel distance, accessibility, or whether every relevant library is represented in OpenStreetMap.

## Completion checklist

- [ ] The required folders are created by code.
- [ ] Raw feature data is saved before cleaning.
- [ ] A later step reopens the saved file.
- [ ] The feature data is inspected before transformation.
- [ ] The spatial join is checked.
- [ ] Missing counts are handled deliberately.
- [ ] The adjusted measure is calculated.
- [ ] A map is produced.
- [ ] Processed spatial data and a CSV summary are saved.
- [ ] The notebook runs from top to bottom.